# Personal Finance Analyser — Python Analysis

## Overview
This notebook performs deeper analysis on the cleaned transactions dataset using pandas. 

## Analysis Sections
1. Monthly spending trends
2. Category breakdown and percentages
3. Rolling averages
4. Month on month change
5. Spending projections

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load the cleaned dataset
df = pd.read_csv('../data/transactions_cleaned.csv', parse_dates=['date'])

# Separate income and expenses
expenses = df[df['type'] == 'expense'].copy()
income = df[df['type'] == 'income'].copy()

print(f"Total transactions: {len(df)}")
print(f"Expense transactions: {len(expenses)}")
print(f"Income transactions: {len(income)}")
print(f"\nDate range: {df['date'].min().strftime('%d %b %Y')} to {df['date'].max().strftime('%d %b %Y')}")

Total transactions: 575
Expense transactions: 543
Income transactions: 32

Date range: 01 Jan 2024 to 30 Dec 2024


In [2]:
# Section 1 - Monthly spending trends
monthly_expenses = expenses.groupby(['month', 'month_name'])['amount'].sum().reset_index()
monthly_expenses.columns = ['month', 'month_name', 'total_spent']
monthly_expenses = monthly_expenses.sort_values('month')

monthly_income = income.groupby(['month', 'month_name'])['amount'].sum().reset_index()
monthly_income.columns = ['month', 'month_name', 'total_income']
monthly_income = monthly_income.sort_values('month')

# Merge into one summary table
monthly_summary = monthly_expenses.merge(monthly_income, on=['month', 'month_name'])
monthly_summary['net_position'] = monthly_summary['total_income'] - monthly_summary['total_spent']
monthly_summary['savings_rate'] = (monthly_summary['net_position'] / monthly_summary['total_income'] * 100).round(2)

print("=== Monthly Summary ===")
print(monthly_summary[['month_name', 'total_income', 'total_spent', 'net_position', 'savings_rate']].to_string(index=False))

=== Monthly Summary ===
month_name  total_income  total_spent  net_position  savings_rate
   January       3172.04      3390.29       -218.25         -6.88
  February       2200.00      4369.40      -2169.40        -98.61
     March       2200.00      6007.72      -3807.72       -173.08
     April       2333.23      5113.26      -2780.03       -119.15
       May       2650.26      6106.68      -3456.42       -130.42
      June       2200.00      5624.67      -3424.67       -155.67
      July       2256.17      5357.40      -3101.23       -137.46
    August       2753.58      5137.69      -2384.11        -86.58
 September       2552.72      4092.51      -1539.79        -60.32
   October       2596.24      5868.76      -3272.52       -126.05
  November       3117.28      4829.23      -1711.95        -54.92
  December       2249.96      4512.16      -2262.20       -100.54


In [3]:
# Section 2 - Category breakdown and percentages
total_spent = expenses['amount'].sum()

category_breakdown = expenses.groupby('category')['amount'].agg(['sum', 'count']).reset_index()
category_breakdown.columns = ['category', 'total_spent', 'num_transactions']
category_breakdown['percentage'] = (category_breakdown['total_spent'] / total_spent * 100).round(2)
category_breakdown = category_breakdown.sort_values('total_spent', ascending=False)

print(f"Total expenses: £{total_spent:,.2f}")
print(f"\n=== Spending by Category ===")
print(category_breakdown.to_string(index=False))

Total expenses: £60,409.77

=== Spending by Category ===
     category  total_spent  num_transactions  percentage
      Savings     19051.91                70       31.54
      Housing     18776.80                54       31.08
     Clothing      5356.83                68        8.87
    Transport      5053.15                67        8.36
         Food      5034.20                74        8.33
Entertainment      2622.80                73        4.34
     Personal      2340.85                83        3.87
       Health      2173.23                54        3.60


In [4]:
# Section 3 - Rolling averages
# Group expenses by date first
daily_expenses = expenses.groupby('date')['amount'].sum().reset_index()
daily_expenses.columns = ['date', 'daily_total']

# Sort by date
daily_expenses = daily_expenses.sort_values('date')

# Calculate 7 day and 30 day rolling averages
daily_expenses['rolling_7day'] = daily_expenses['daily_total'].rolling(window=7).mean().round(2)
daily_expenses['rolling_30day'] = daily_expenses['daily_total'].rolling(window=30).mean().round(2)

print("=== Daily Expenses with Rolling Averages (last 10 days) ===")
print(daily_expenses.tail(10).to_string(index=False))
print(f"\nOverall average daily spend: £{daily_expenses['daily_total'].mean():.2f}")
print(f"7 day rolling average (final): £{daily_expenses['rolling_7day'].iloc[-1]:.2f}")
print(f"30 day rolling average (final): £{daily_expenses['rolling_30day'].iloc[-1]:.2f}")

=== Daily Expenses with Rolling Averages (last 10 days) ===
      date  daily_total  rolling_7day  rolling_30day
2024-12-19       477.11        291.50         181.98
2024-12-20       804.53        339.15         206.70
2024-12-21       161.58        324.98         202.34
2024-12-23        63.30        316.17         203.79
2024-12-24        64.22        266.40         202.67
2024-12-26        32.75        268.13         200.35
2024-12-27        57.34        237.26         193.97
2024-12-28       150.40        190.59         194.52
2024-12-29        84.33         87.70         196.32
2024-12-30       213.98         95.19         197.57

Overall average daily spend: £223.74
7 day rolling average (final): £95.19
30 day rolling average (final): £197.57


In [5]:
# Section 4 - Month on month change
monthly_expenses_simple = expenses.groupby(['month', 'month_name'])['amount'].sum().reset_index()
monthly_expenses_simple.columns = ['month', 'month_name', 'total_spent']
monthly_expenses_simple = monthly_expenses_simple.sort_values('month')

# Calculate month on month change
monthly_expenses_simple['prev_month'] = monthly_expenses_simple['total_spent'].shift(1)
monthly_expenses_simple['mom_change'] = (monthly_expenses_simple['total_spent'] - monthly_expenses_simple['prev_month']).round(2)
monthly_expenses_simple['mom_pct_change'] = ((monthly_expenses_simple['mom_change'] / monthly_expenses_simple['prev_month']) * 100).round(2)

print("=== Month on Month Spending Change ===")
print(monthly_expenses_simple[['month_name', 'total_spent', 'mom_change', 'mom_pct_change']].to_string(index=False))

=== Month on Month Spending Change ===
month_name  total_spent  mom_change  mom_pct_change
   January      3390.29         NaN             NaN
  February      4369.40      979.11           28.88
     March      6007.72     1638.32           37.50
     April      5113.26     -894.46          -14.89
       May      6106.68      993.42           19.43
      June      5624.67     -482.01           -7.89
      July      5357.40     -267.27           -4.75
    August      5137.69     -219.71           -4.10
 September      4092.51    -1045.18          -20.34
   October      5868.76     1776.25           43.40
  November      4829.23    -1039.53          -17.71
  December      4512.16     -317.07           -6.57


In [6]:
# Section 5 - Spending projections
# Use numpy to fit a simple linear trend to monthly expenses
months = monthly_expenses_simple['month'].values
spent = monthly_expenses_simple['total_spent'].values

# Fit a linear trend line
coefficients = np.polyfit(months, spent, 1)
slope = coefficients[0]
intercept = coefficients[1]

print(f"=== Spending Trend Analysis ===")
print(f"Monthly trend: £{slope:+.2f} per month")
print(f"(Positive = spending increasing, Negative = spending decreasing)")

# Project next 3 months
print(f"\n=== 3 Month Projection (Jan-Mar 2025) ===")
for month_num in [13, 14, 15]:
    projected = slope * month_num + intercept
    month_name = ['January', 'February', 'March'][month_num - 13]
    print(f"{month_name} 2025: £{projected:,.2f} projected spending")

print(f"\nAverage monthly spending 2024: £{spent.mean():,.2f}")
print(f"Average monthly income 2024: £{monthly_summary['total_income'].mean():,.2f}")
print(f"Average monthly deficit 2024: £{(monthly_summary['total_income'] - monthly_summary['total_spent']).mean():,.2f}")

=== Spending Trend Analysis ===
Monthly trend: £+25.27 per month
(Positive = spending increasing, Negative = spending decreasing)

=== 3 Month Projection (Jan-Mar 2025) ===
January 2025: £5,198.43 projected spending
February 2025: £5,223.70 projected spending
March 2025: £5,248.98 projected spending

Average monthly spending 2024: £5,034.15
Average monthly income 2024: £2,523.46
Average monthly deficit 2024: £-2,510.69
